<a href="https://colab.research.google.com/github/hadiritch-cell/Strogatz/blob/main/Image_Source_Pentagon_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
=============================================================================
Image Source Method (ISM) Validator for Irregular Pentagon
=============================================================================

PURPOSE:
    Validate the image source positions computed by MATLAB using Python.

LEARNING GOALS:
    1. Understand the reflection formula (Cuenca et al., 2014, Eq. 50)
    2. Implement BFS tree traversal for image source generation
    3. Compare Python results with MATLAB output

REFLECTION FORMULA:
    Given a source at position r_m, its reflection across an edge
    defined by vertices v_p and v_{p+1} is:

    r_image = 2 * foot - r_m

    where 'foot' is the perpendicular projection of r_m onto the edge line.

    Mathematically:
        edge = v_{p+1} - v_p
        t = [(r_m - v_p) · edge] / |edge|²
        foot = v_p + t * edge
        r_image = 2 * foot - r_m

=============================================================================
"""

import numpy as np
from collections import deque  # For BFS queue


# =============================================================================
# STAGE 1: THE REFLECTION FUNCTION
# =============================================================================

def reflect_point(r_m, v_p, v_p1):
    """
    Reflect a point r_m across the line defined by vertices v_p and v_p1.

    This implements Equation (50) from Cuenca et al. (2014).

    Parameters
    ----------
    r_m : np.ndarray, shape (2,)
        The source position to reflect [x, y]
    v_p : np.ndarray, shape (2,)
        First vertex of the edge [x, y]
    v_p1 : np.ndarray, shape (2,)
        Second vertex of the edge [x, y]

    Returns
    -------
    np.ndarray, shape (2,)
        The reflected (image) source position

    Geometric Interpretation
    ------------------------
    1. Find the direction vector of the edge: edge = v_p1 - v_p
    2. Project r_m onto this edge line to find the "foot" point
    3. The image is on the opposite side of the edge, same distance from foot

    ASCII Diagram:

        r_m (source)
         |
         |  (perpendicular)
         |
        foot -------- edge line (v_p to v_p1)
         |
         |
        r_image (reflected)
    """
    # Step 1: Edge direction vector
    edge = v_p1 - v_p

    # Step 2: Parameter t for projection onto edge line
    # t = (r_m - v_p) · edge / |edge|²
    # This finds how far along the edge the foot point lies
    t = np.dot(r_m - v_p, edge) / np.dot(edge, edge)

    # Step 3: Foot of perpendicular (closest point on edge line to r_m)
    foot = v_p + t * edge

    # Step 4: Reflect r_m through the foot
    # The image is at the same distance from foot, but on the opposite side
    r_image = 2 * foot - r_m

    return r_image


# =============================================================================
# STAGE 2: SET UP POLYGON AND DATA STRUCTURES
# =============================================================================

def setup_pentagon():
    """
    Define the irregular pentagon vertices and original source.

    Returns the same values as the MATLAB code for validation.

    Note: MATLAB uses column vectors, Python/NumPy uses row vectors by default.
    We'll store vertices as rows for easier indexing: V[i] gives vertex i.
    """
    # Pentagon vertices (matching MATLAB's V matrix, but transposed)
    # MATLAB: V = [1.4564, 0.87, 0.00, 0.00, 0.69;
    #              0.40381, 1.1027, 0.6993, 0.2993, -0.1328]
    # Each row here is one vertex [x, y]
    V = np.array([
        [1.4564,  0.40381],   # V_1
        [0.87,    1.1027 ],   # V_2
        [0.00,    0.6993 ],   # V_3
        [0.00,    0.2993 ],   # V_4
        [0.69,   -0.1328 ]    # V_5
    ])

    # Original source position (inside the pentagon)
    S0 = np.array([0.6033, 0.4745])

    return V, S0


# =============================================================================
# STAGE 3: BUILD IMAGE SOURCE TREE (BFS)
# =============================================================================

class ImageSource:
    """
    Data structure to store information about each image source.

    Attributes
    ----------
    pos : np.ndarray
        Position [x, y] of this image source
    order : int
        Reflection order (0 = original source, 1 = first reflection, etc.)
    last_edge : int
        Index of the edge last reflected across (0 = none for original)
    parent_idx : int
        Index of the parent source in the sources list
    edge_chain : list
        Full sequence of edges reflected across (for labeling)
    """
    def __init__(self, pos, order, last_edge, parent_idx, edge_chain):
        self.pos = pos
        self.order = order
        self.last_edge = last_edge
        self.parent_idx = parent_idx
        self.edge_chain = edge_chain.copy()  # Important: copy the list!

    def get_label(self):
        """Generate label like S_{0,2,4,1} based on edge chain."""
        if self.order == 0:
            return "S_0"
        chain_str = ",".join(str(e) for e in self.edge_chain)
        return f"S_{{0,{chain_str}}}"


def build_image_sources(V, S0, max_order):
    """
    Build the complete image source tree using BFS traversal.

    This mirrors the MATLAB implementation exactly:
    - Start with the original source (order 0)
    - For each source, reflect across all edges except the "parent" edge
    - Continue until max_order is reached

    Parameters
    ----------
    V : np.ndarray, shape (N, 2)
        Polygon vertices
    S0 : np.ndarray, shape (2,)
        Original source position
    max_order : int
        Maximum reflection order

    Returns
    -------
    list of ImageSource
        All image sources (including original)

    Why skip the parent edge?
    -------------------------
    If source S was created by reflecting across edge p, reflecting S
    back across edge p would recover the parent source. This creates
    a "back-reflection" that we must avoid.
    """
    N = len(V)  # Number of vertices = number of edges

    # Initialize with original source (order 0)
    sources = [ImageSource(
        pos=S0.copy(),
        order=0,
        last_edge=0,       # No parent edge for original
        parent_idx=-1,     # No parent
        edge_chain=[]      # Empty chain
    )]

    # BFS queue: stores indices into the sources list
    # We use a simple index-based approach (like MATLAB's while loop)
    read_idx = 0  # Current source being processed

    while read_idx < len(sources):
        current = sources[read_idx]

        # Only generate children if below max_order
        if current.order < max_order:

            # Try reflecting across each edge
            for edge in range(1, N + 1):  # Edges numbered 1 to N (like MATLAB)

                # Skip back-reflection across parent edge
                if edge == current.last_edge:
                    continue

                # Get edge vertices
                # Edge p connects V[p-1] to V[p % N] (0-indexed in Python)
                v_p = V[edge - 1]           # First vertex of edge
                v_p1 = V[edge % N]          # Second vertex (wraps around)

                # Compute reflected position
                new_pos = reflect_point(current.pos, v_p, v_p1)

                # Build edge chain for this new source
                new_chain = current.edge_chain + [edge]

                # Create and store new image source
                new_source = ImageSource(
                    pos=new_pos,
                    order=current.order + 1,
                    last_edge=edge,
                    parent_idx=read_idx,
                    edge_chain=new_chain
                )
                sources.append(new_source)

        read_idx += 1

    return sources


# =============================================================================
# STAGE 4: DISPLAY AND COMPARE RESULTS
# =============================================================================

def print_results(sources, max_order):
    """
    Print image source positions grouped by order.

    Format matches MATLAB output for easy visual comparison.
    """
    print("=" * 60)
    print("PYTHON VALIDATION: Image Source Positions")
    print("=" * 60)
    print()

    for order in range(1, max_order + 1):
        # Find all sources of this order
        order_sources = [s for s in sources if s.order == order]

        print(f"Order-{order} image sources  ({len(order_sources)} sources)")
        print("-" * 40)
        print(f"{'Source':<16}  {'x':>10}  {'y':>10}")
        print("-" * 40)

        for src in order_sources:
            label = src.get_label()
            print(f"{label:<16}  {src.pos[0]:>+10.4f}  {src.pos[1]:>+10.4f}")

        print("-" * 40)
        print()

    # Total count (excluding order 0)
    total = sum(1 for s in sources if s.order > 0)
    print(f"Total image sources (orders 1 to {max_order}): {total}")
    print()


def count_sources_per_order(sources, max_order):
    """Show source count breakdown by order."""
    print("Source count by order:")
    print("-" * 30)
    for order in range(max_order + 1):
        count = sum(1 for s in sources if s.order == order)
        print(f"  Order {order}: {count} sources")
    print("-" * 30)
    print()


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """
    Main function to run the validation.

    Steps:
    1. Set up the pentagon (same as MATLAB)
    2. Build image sources using BFS
    3. Print results for comparison
    """
    print("\n" + "=" * 60)
    print("IMAGE SOURCE METHOD - PYTHON VALIDATOR")
    print("=" * 60 + "\n")

    # Configuration (must match MATLAB)
    max_order = 5

    # Stage 2: Set up polygon
    V, S0 = setup_pentagon()
    N = len(V)

    print("Pentagon Vertices:")
    print("-" * 35)
    for i in range(N):
        print(f"  V_{i+1}: ({V[i, 0]:+.4f}, {V[i, 1]:+.4f})")
    print("-" * 35)
    print(f"\nOriginal source S_0: ({S0[0]:.4f}, {S0[1]:.4f})")
    print(f"Maximum order: {max_order}")
    print()

    # Stage 3: Build image source tree
    print("Building image source tree...")
    sources = build_image_sources(V, S0, max_order)
    print(f"Done! Generated {len(sources)} total sources.\n")

    # Show breakdown
    count_sources_per_order(sources, max_order)

    # Stage 4: Print detailed results
    print_results(sources, max_order)

    # ---------------------------------------------------------------------
    # VALIDATION CHECKPOINT
    # ---------------------------------------------------------------------
    # Expected counts from MATLAB (N=5 pentagon):
    #   Order 0: 1 source (original)
    #   Order 1: 5 sources (reflect across each of 5 edges)
    #   Order 2: 5 * 4 = 20 sources (each order-1 reflects across 4 edges)
    #   Order 3: 20 * 4 = 80 sources
    #   Order 4: 80 * 4 = 320 sources
    #   Order 5: 320 * 4 = 1280 sources
    #   Total (orders 1-5): 5 + 20 + 80 + 320 + 1280 = 1705

    print("=" * 60)
    print("VALIDATION CHECKPOINT")
    print("=" * 60)
    expected_counts = {
        0: 1,
        1: 5,
        2: 20,
        3: 80,
        4: 320,
        5: 1280
    }

    all_match = True
    for order, expected in expected_counts.items():
        actual = sum(1 for s in sources if s.order == order)
        status = "✓" if actual == expected else "✗"
        if actual != expected:
            all_match = False
        print(f"  Order {order}: expected {expected}, got {actual} {status}")

    print()
    if all_match:
        print("✓ All source counts match expected values!")
    else:
        print("✗ Some counts don't match - check implementation")
    print()


if __name__ == "__main__":
    main()


IMAGE SOURCE METHOD - PYTHON VALIDATOR

Pentagon Vertices:
-----------------------------------
  V_1: (+1.4564, +0.4038)
  V_2: (+0.8700, +1.1027)
  V_3: (+0.0000, +0.6993)
  V_4: (+0.0000, +0.2993)
  V_5: (+0.6900, -0.1328)
-----------------------------------

Original source S_0: (0.6033, 0.4745)
Maximum order: 5

Building image source tree...
Done! Generated 1706 total sources.

Source count by order:
------------------------------
  Order 0: 1 sources
  Order 1: 5 sources
  Order 2: 20 sources
  Order 3: 80 sources
  Order 4: 320 sources
  Order 5: 1280 sources
------------------------------

PYTHON VALIDATION: Image Source Positions

Order-1 image sources  (5 sources)
----------------------------------------
Source                     x           y
----------------------------------------
S_{0,1}              +1.5350     +1.2562
S_{0,2}              +0.2182     +1.3050
S_{0,3}              -0.6033     +0.4745
S_{0,4}              +0.1058     -0.3200
S_{0,5}              +1.2310  